In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.csv('/Volumes/workspace/default/firstvolume/BigMart Sales.csv', header=True, inferSchema=True)
df.display()

# Pivot

In [0]:
df.groupBy("Item_Type").pivot("Outlet_Size").agg(avg("Item_MRP")).display()

# When-Otherwise
This function is similar to case-when statements in sql

In [0]:
df.withColumn('Food_flag', when(col("Item_Type").isin("Meat", "Seafood"), "Non-Veg").otherwise("Veg")).display()

In [0]:
df.withColumn('Food_Flag', when(col("Item_Type").isin("Meat", "Seafood"), "Non-Veg").otherwise(when(col("Item_MRP")>100, "Veg-Expensive").otherwise("Veg-InExpensive"))).display()

# Joins
1. Inner join
2. Left join
3. Right join
4. Full join
5. Anti join

In [0]:
data1 = [('1', 'Alexa','d01'),
         ('2', 'Bravo','d02'),
         ('3', 'Charlie','d03'),
         ('4', 'Delta','d03'),
         ('5', 'Echo','d05'),
         ('6', 'Foxtrot','d06')]
schema1 = 'emp_id string, emp_name string, dept_id string'
df1 = spark.createDataFrame(data1, schema1)

data2 = [('d01', 'IT'),
         ('d02', 'HR'),
         ('d03', 'Marketing'),
         ('d04', 'Sales'),
         ('d05', 'Finance')]
schema2 = 'dept_id string, dept_name string'
df2 = spark.createDataFrame(data2, schema2)
df1.display()
df2.display()

### Inner join

In [0]:
df1.join(df2, df1['dept_id'] == df2['dept_id'],'inner').display()

### Left join

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'], "left").display()

### Right join

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'], "right").display()

### Anti-join

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'], "anti").display()

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'], "full").display()

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'], "cross").display()

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'], "full_outer").display()

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'], "left_anti").display()

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'], "left_outer").display()

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'], "left_semi").display()

# Window functions

### Row_Number
Row Number gives an unique number to the rows from 1 to n and it does not exclude duplicates and keep it as such

In [0]:
from pyspark.sql.window import *

In [0]:
df.display()

In [0]:
df.withColumn('Row_Number', row_number().over(Window.orderBy('Item_Identifier'))).display()

### Rank and Dense_Rank
- Rank -> Rank assigns the same rank to all the same values but after that it ranks the next value with the number skipping the ranks inbetween. Eg: if 6 values are same and they get rank 1, then the 7th value get the rank 7
- Dense_rank -> It works the same as rank except it continues the rank number. Eg: if 6 values has rank 1 then 7th value has rank 2
![image_1793973929913.png](./image_1793973929913.png "image_1793973929913.png")

In [0]:
df.withColumns({'Rank': rank().over(Window.orderBy('Item_Identifier')), 'Dense_Rank': dense_rank().over(Window.orderBy('Item_identifier'))}).display()

### Cumulative sum
Sum over the rows like sql

In [0]:
df.withColumn('cumulative_sum', sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding, Window.currentRow))).display()

In [0]:
df.withColumn('cumulative_sum', sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))).display()

# User Defined Function - UDF
Try to avoid using UDF since it causes the performance issue and use builtin pyspark mostly.
![image_1793981566226.png](./image_1793981566226.png "image_1793981566226.png")

#### Step 1

In [0]:
def my_func(x):
    return x*x

#### Step 2

In [0]:
my_udf = udf(my_func)

#### Step 3

In [0]:
df.withColumn('udf_value', my_udf('Item_MRP')).display()